# ML Student Performance Prediction and Model Evaluation – Day 18 & 19
Complete EDA, regression, classification, evaluation, and model analysis.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, confusion_matrix, ConfusionMatrixDisplay, accuracy_score, precision_score, recall_score, f1_score, classification_report

df = pd.read_csv("Day18_19_student_habits_performance.csv")
display(df.head())

## 1. Dataset Inspection

In [ ]:
print("Shape:", df.shape)
display(df.info())
display(df.describe(include="all").T)
print("\nMissing values:")
display(df.isnull().sum())

## 2. Numerical and Categorical Variables

In [ ]:
numerical_cols = df.select_dtypes(include=np.number).columns.tolist()
categorical_cols = df.select_dtypes(exclude=np.number).columns.tolist()
print("Numerical:", numerical_cols)
print("Categorical:", categorical_cols)

## 3. Distributions and Outliers

In [ ]:
analysis_numeric = [c for c in numerical_cols if c != "student_id"]
df[analysis_numeric].hist(figsize=(14,10), bins=25)
plt.tight_layout()
plt.show()

plt.figure(figsize=(12,6))
sns.boxplot(data=df[analysis_numeric])
plt.xticks(rotation=45)
plt.title("Boxplots for Potential Outliers")
plt.tight_layout()
plt.show()

## 4. Relationships and Correlations with Exam Score

In [ ]:
corr = df[numerical_cols].corr()
plt.figure(figsize=(10,8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation Matrix")
plt.tight_layout()
plt.show()

print("Correlation with exam_score:")
display(corr["exam_score"].sort_values(ascending=False).to_frame("Correlation"))

In [ ]:
features_to_plot = ["study_hours_per_day","attendance_percentage","sleep_hours","social_media_hours","netflix_hours","exercise_frequency","mental_health_rating"]
for col in features_to_plot:
    plt.figure(figsize=(6,4))
    sns.scatterplot(data=df, x=col, y="exam_score")
    plt.title(f"{col} vs exam_score")
    plt.tight_layout()
    plt.show()

## 5. Regression: Predict Exam Score

In [ ]:
target = "exam_score"
features = [c for c in df.columns if c not in ["student_id", target]]
X = df[features]
y = df[target]

num_features = X.select_dtypes(include=np.number).columns.tolist()
cat_features = X.select_dtypes(exclude=np.number).columns.tolist()

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])
preprocessor = ColumnTransformer([
    ("num", numeric_transformer, num_features),
    ("cat", categorical_transformer, cat_features)
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

regression_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LinearRegression())
])

regression_model.fit(X_train, y_train)
train_pred = regression_model.predict(X_train)
test_pred = regression_model.predict(X_test)

reg_results = pd.DataFrame({
    "Metric": ["MAE","RMSE","R²"],
    "Training": [
        mean_absolute_error(y_train, train_pred),
        mean_squared_error(y_train, train_pred, squared=False),
        r2_score(y_train, train_pred)
    ],
    "Testing": [
        mean_absolute_error(y_test, test_pred),
        mean_squared_error(y_test, test_pred, squared=False),
        r2_score(y_test, test_pred)
    ]
})
display(reg_results)

### Regression Interpretation
MAE and RMSE measure prediction error; lower is better. R² measures explained variance; higher is better. Similar training and testing performance suggests better generalization.

In [ ]:
plt.figure(figsize=(6,5))
sns.scatterplot(x=y_test, y=test_pred)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], linestyle="--")
plt.xlabel("Actual Exam Score")
plt.ylabel("Predicted Exam Score")
plt.title("Actual vs Predicted Exam Scores")
plt.tight_layout()
plt.show()

## 6. Classification: Pass/Fail

In [ ]:
y_class = (df["exam_score"] >= 50).astype(int)
print("Class mapping: 1 = Pass, 0 = Fail")
print(y_class.value_counts(normalize=True).rename("Proportion"))

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X, y_class, test_size=0.2, random_state=42, stratify=y_class
)

classification_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=2000))
])

classification_model.fit(X_train_c, y_train_c)
train_class_pred = classification_model.predict(X_train_c)
test_class_pred = classification_model.predict(X_test_c)

class_results = pd.DataFrame({
    "Metric": ["Accuracy","Precision","Recall","F1-score"],
    "Training": [
        accuracy_score(y_train_c, train_class_pred),
        precision_score(y_train_c, train_class_pred, zero_division=0),
        recall_score(y_train_c, train_class_pred, zero_division=0),
        f1_score(y_train_c, train_class_pred, zero_division=0)
    ],
    "Testing": [
        accuracy_score(y_test_c, test_class_pred),
        precision_score(y_test_c, test_class_pred, zero_division=0),
        recall_score(y_test_c, test_class_pred, zero_division=0),
        f1_score(y_test_c, test_class_pred, zero_division=0)
    ]
})
display(class_results)
print(classification_report(y_test_c, test_class_pred, target_names=["Fail","Pass"]))

In [ ]:
cm = confusion_matrix(y_test_c, test_class_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Fail","Pass"])
disp.plot()
plt.title("Classification Confusion Matrix")
plt.show()

## 7. Training vs Testing Performance and Overfitting Analysis

In [ ]:
train_r2 = r2_score(y_train, train_pred)
test_r2 = r2_score(y_test, test_pred)
train_acc = accuracy_score(y_train_c, train_class_pred)
test_acc = accuracy_score(y_test_c, test_class_pred)

print("Regression R² - Training:", round(train_r2, 4))
print("Regression R² - Testing:", round(test_r2, 4))
print("Classification Accuracy - Training:", round(train_acc, 4))
print("Classification Accuracy - Testing:", round(test_acc, 4))

if train_r2 - test_r2 > 0.10 or train_acc - test_acc > 0.10:
    print("\nPossible sign of overfitting: training performance is substantially better than testing performance.")
elif train_r2 < 0.3 and test_r2 < 0.3:
    print("\nPossible underfitting: both training and testing regression performance are weak.")
else:
    print("\nNo strong evidence of severe overfitting based on the training/testing comparison.")

## 8. Important Findings and 5 Insights

In [ ]:
corr_exam = corr["exam_score"].drop("exam_score").sort_values(key=abs, ascending=False)
print("Top numerical relationships with exam_score:")
display(corr_exam.head(8).to_frame("Correlation"))

print("\nFive meaningful insights:")
print("1. The strongest numerical relationships with exam_score are shown above.")
print("2. Study habits, attendance, sleep, entertainment time, exercise, and mental health can be compared using correlation and model results.")
print("3. Regression performance indicates how accurately the selected features predict the continuous exam score.")
print("4. Classification metrics show how effectively the model identifies students likely to pass or fail.")
print("5. Comparing training and testing metrics helps detect possible overfitting and evaluate model generalization.")

## Conclusion
This notebook follows a complete machine-learning workflow: EDA, preprocessing, Linear Regression for exam-score prediction, Logistic Regression for Pass/Fail prediction, evaluation with regression and classification metrics, and training-versus-testing performance analysis.